### **Dataset Generation**

In [1]:
# Toy dataset for regression.
# last column is the label and first two columns are features
training_data = [
    [15, 60, 60.0],
    [20, 50, 65.0],
    [10, 70, 55.0],
    [25, 40, 70.0],
    [18, 55, 63.5],
    [12, 65, 56.5],
]

# Column labels.
header = ["temp", "humidity", "value"]

### **Unique Values**

In [2]:
# Find the unique values for a column in a dataset
def unique_vals(rows, col):
    return set([row[col] for row in rows])

In [3]:
unique_vals(training_data, 0)

{10, 12, 15, 18, 20, 25}

In [ ]:
unique_vals(training_data, 1)

{40, 50, 55, 60, 65, 70}

In [ ]:
def value_counts(rows):
    """Counts the number of each type of example in a dataset."""
    counts = {}  # a dictionary of value -> count.
    for row in rows:
        # in our dataset format, the value is always the last column
        label = row[-1]
        if label not in counts:
            counts[label] = 0
        counts[label] += 1
    return counts


In [ ]:
#######
# Demo:
value_counts(training_data)
#######

{60.0: 1, 65.0: 1, 55.0: 1, 70.0: 1, 63.5: 1, 56.5: 1}

**

In [4]:
def is_numeric(value):
    """Test if a value is numeric."""
    return isinstance(value, int) or isinstance(value, float)

In [5]:
#######
# Demo:
is_numeric(7)
#######

True

In [6]:
is_numeric("Red")

False

### **Questions**

   A Question is used to partition a dataset.

    This class just records a 'column number' (e.g., 0 for Color) and a
    'column value' (e.g., Green). The 'match' method is used to compare
    the feature value in an example to the feature value stored in the
    question. See the demo below.
    

In [7]:
class Question:

    def __init__(self, column, value):
        self.column = column
        self.value = value

    def match(self, example):
        # Compare the feature value in an example to the
        # feature value in this question.
        val = example[self.column]
        if is_numeric(val):
            return val >= self.value
        else:
            return val == self.value

    def __repr__(self):
        # This is just a helper method to print
        # the question in a readable format.
        condition = "=="
        if is_numeric(self.value):
            condition = ">="
        return "Is %s %s %s?" % (
            header[self.column], condition, str(self.value))


In [8]:
# Let's write a question for a numeric attribute
Question(0, 6.0)

Is temp >= 6.0?

### **Partition**

    For each row in the dataset, check if it matches the question. If
    so, add it to 'true rows', otherwise, add it to 'false rows'

In [13]:
def partition(rows, question):
    true_rows, false_rows = [], []
    for row in rows:
        if question.match(row):
            true_rows.append(row)
        else:
            false_rows.append(row)
    return true_rows, false_rows

In [14]:
# partition the training data based on whether temp is >= 7.0.
true_rows, false_rows = partition(training_data, Question(0, 15.0))

In [15]:
# all the rows with temp >= 15.0.
true_rows

[[15, 60, 60.0], [20, 50, 65.0], [25, 40, 70.0], [18, 55, 63.5]]

In [16]:
false_rows

[[10, 70, 55.0], [12, 65, 56.5]]

### **MSE**

In [17]:
def mse(rows):
    if not rows:
        return 0
    actual_values = [row[-1] for row in rows]
    predicted_value = sum(actual_values) / len(actual_values)
    return sum([(y - predicted_value) ** 2 for y in actual_values]) / len(actual_values)

In [18]:
# a dataset with no variance.
no_variance = [[10], [10], [10]]
# this will return 0
mse(no_variance)

0.0

In [19]:
# a dataset with some variance.
some_variance = [[10], [20]]
# this will return ( (10-15)^2 + (20-15)^2 ) / 2 = 25
mse(some_variance)

25.0

### **Information Gain**



    The uncertainty of the starting node, minus the weighted uncertainty of
    two child nodes.
    

In [20]:
def info_gain(left, right, current_uncertainty):
    p = float(len(left)) / (len(left) + len(right))
    return current_uncertainty - p * mse(left) - (1 - p) * mse(right)

In [21]:
# Calculate the uncertainty of our training data.
current_uncertainty = mse(training_data)

In [22]:
# How much information do we gain by partitioning on temp >= 15?
true_rows, false_rows = partition(training_data, Question(0, 16))
info_gain(true_rows, false_rows, current_uncertainty)

20.250000000000004

In [23]:
# What about if we partitioned on humidity >= 30?
true_rows, false_rows = partition(training_data, Question(1, 60))
info_gain(true_rows, false_rows, current_uncertainty)

20.250000000000004

### **Best Split**

    Find the best question to ask by iterating over every feature / value
    and calculating the information gain

In [25]:
def find_best_split(rows):
    best_gain = 0  # keep track of the best information gain
    best_question = None  # keep track of the feature / value that produced it
    current_uncertainty = mse(rows)
    n_features = len(rows[0]) - 1  # number of columns

    for col in range(n_features):  # for each feature

        values = set([row[col] for row in rows])  # unique values in the column

        for val in values:

            question = Question(col, val)

            # try splitting the dataset
            true_rows, false_rows = partition(rows, question)

            # Skip this split if it doesn't divide the dataset.
            if len(true_rows) == 0 or len(false_rows) == 0:
                continue

            # Calculate the information gain from this split
            gain = info_gain(true_rows, false_rows, current_uncertainty)

            if gain >= best_gain:
                best_gain, best_question = gain, question

    return best_gain, best_question

In [26]:
# Find the best question to ask first for our toy dataset.
best_gain, best_question = find_best_split(training_data)

In [27]:
best_gain

20.250000000000004

In [28]:
best_question

Is humidity >= 60?

### **Leaf Class**

In [31]:
class Leaf:
    """A Leaf node predicts a value.

    This holds the average value of the rows from the training data that
    reach this leaf.
    """

    def __init__(self, rows):
        self.prediction = sum([row[-1] for row in rows]) / len(rows)

### **Decision Node**

In [32]:
class Decision_Node:
    """A Decision Node asks a question.

    This holds a reference to the question, and to the two child nodes.
    """

    def __init__(self,
                 question,
                 true_branch,
                 false_branch):
        self.question = question
        self.true_branch = true_branch
        self.false_branch = false_branch

### **Build Tree**

In [33]:
def build_tree(rows):

    gain, question = find_best_split(rows)

    if gain == 0:
        return Leaf(rows)

    true_rows, false_rows = partition(rows, question)

    true_branch = build_tree(true_rows)

    false_branch = build_tree(false_rows)

    return Decision_Node(question, true_branch, false_branch)

### **Print Tree**

In [34]:
def print_tree(node, spacing=""):

    # Base case: we've reached a leaf
    if isinstance(node, Leaf):
        print (spacing + "Predict", node.prediction)
        return

    # Print the question at this node
    print (spacing + str(node.question))

    # Call this function recursively on the true branch
    print (spacing + '--> True:')
    print_tree(node.true_branch, spacing + "  ")

    # Call this function recursively on the false branch
    print (spacing + '--> False:')
    print_tree(node.false_branch, spacing + "  ")

### **Classify**

In [35]:
def classify(row, node):

    # Base case: we've reached a leaf
    if isinstance(node, Leaf):
        return node.prediction

    if node.question.match(row):
        return classify(row, node.true_branch)
    else:
        return classify(row, node.false_branch)

In [37]:
# The tree predicts the 1st row of our training data.
my_tree = build_tree(training_data)
classify(training_data[0], my_tree)

60.0

### **Evaluation**

In [38]:
# Evaluate
testing_data = [
    [16, 62, 63.0],
    [22, 53, 70.5],
    [9, 68, 51.0],
    [26, 42, 73.0],
    [19, 57, 66.5],
]

In [39]:
for row in testing_data:
        print ("Actual: %s. Predicted: %s" %
               (row[-1], classify(row, my_tree)))

Actual: 63.0. Predicted: 60.0
Actual: 70.5. Predicted: 65.0
Actual: 51.0. Predicted: 56.5
Actual: 73.0. Predicted: 70.0
Actual: 66.5. Predicted: 63.5
